# 1. Dataset Preparation

Based on: https://github.com/kerrgarr/SemanticSegmentationCityscapes

Download Cityscapes datasets: https://www.cityscapes-dataset.com/downloads

Datasets used: gtFine_trainvaltest.zip (241MB) and leftmg8bit_trainvaltest.zip (11GB).

Unzip both files in the data/ directory.

Your data/ folder structure should look like:

data/   
├──gtFine_trainvaltest/   
└──leftImg8bit_trainvaltest/    

In [ ]:
import os
import shutil
from tqdm import tqdm

New dataset directory

In [ ]:
dataset_dir = './data/dataset/'

Unzipped cityscapes data paths

In [ ]:
gtFineDIR = './data/gtFine_trainvaltest/gtFine/'
ImgDIR = './data/leftImg8bit_trainvaltest/leftImg8bit/'

The Cityscapes test split does not contian image masks, only dummy blanks, so it cannot be used to evaluate model accuracy. 

Use some data from the original train split to create test split, but exclude it.

In [ ]:
# Train directory
gtFineDIR_train = os.path.join(gtFineDIR, 'train')
ImgDIR_train = os.path.join(ImgDIR, 'train')

dataset_train = os.path.join(dataset_dir,'train')

# Validation directory
gtFineDIR_val = os.path.join(gtFineDIR, 'val')
ImgDIR_val = os.path.join(ImgDIR, 'val')

dataset_valid = os.path.join(dataset_dir,'val')

# Test directory
dataset_test = os.path.join(dataset_dir, 'test')

Function to copy files

In [ ]:
# copy images and labels per cities into one target directory
def create_splits(sourcedir, targetdir, cities_list):
    extensions = ['labelIds', 'leftImg8bit'] # only labels and images needed
    os.makedirs(targetdir, exist_ok=True)
    
    for city in tqdm(cities_list, desc="Cities"):
        directory_city = os.path.join(sourcedir, city)
        images = os.listdir(directory_city)
        for image in images:
            if any(word in image for word in extensions):
                image_path = os.path.join(directory_city, image)
                try:
                    shutil.copy(image_path, targetdir)
                except OSError as e:
                    tqdm.write(str(e))

In [ ]:
# See available cities in the dataset
for folder in os.listdir(ImgDIR):
    cities = []
    for city in os.listdir(os.path.join(ImgDIR, folder)):
        cities.append(city)
    print(f"{folder}: {cities}")

In [ ]:
# See how many images are in each city folder
for folder in os.listdir(ImgDIR):
    print(folder)
    for city in os.listdir(os.path.join(ImgDIR, folder)):
        path = os.path.join(ImgDIR, folder, city)
        print(f"  {city}: {len(os.listdir(path))}")

Select cities to use in each split.

We keep entire cities data per split instead of randomly selecting images from all cities. This is to prevent data leakage from similar looking frames and to improve generalization.

In [ ]:
train_cities = ['aachen', 'bochum', 'bremen', 'cologne', 'darmstadt', 'erfurt', 'hamburg', 'hanover', 'krefeld', 'monchengladbach', 'strasbourg', 'tubingen', 'ulm', 'weimar', 'zurich']

create_splits(gtFineDIR_train, dataset_train, train_cities)
create_splits(ImgDIR_train, dataset_train, train_cities)

In [ ]:
valid_cities = ['frankfurt', 'lindau', 'munster']

create_splits(gtFineDIR_val, dataset_valid, valid_cities)
create_splits(ImgDIR_val, dataset_valid, valid_cities)

In [ ]:
# NOTE - These cities are from the original train split. We do not include them in our train split!
test_cities = ['dusseldorf', 'jena', 'stuttgart']

create_splits(gtFineDIR_train, dataset_test, test_cities)
create_splits(ImgDIR_train, dataset_test, test_cities)

In [ ]:
# Roughly 70 / 15 / 15 split
for folder in os.listdir(dataset_dir):
    path = os.path.join(dataset_dir, folder)
    print(folder, len(os.listdir(path))//2) # divide by 2 because each image has a matching mask file